### Data Loader

In [ ]:
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image, ImageReadMode
from torchvision.transforms import v2

PATCH = 14
EXTS = {".png", ".jpg", ".jpeg"}


class RGBDataset(Dataset):
    """
    Returns per frame:
      rgb   (3, H, W) float32 in [0, 1], H and W multiples of 14
      idx   position in the sorted file list
      name  file stem (use it to match poses / BEV patches / labels)
    """

    def __init__(self, folder, out_hw=(364, 644), train=False):
        assert out_hw[0] % PATCH == 0 and out_hw[1] % PATCH == 0, "out_hw must be multiples of 14"
        self.files = sorted(p for p in Path(folder).iterdir() if p.suffix.lower() in EXTS)
        assert self.files, f"no images found in {folder}"
        self.out_hw = out_hw
        self.train = train
        self.jitter = v2.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.03)
        self.blur = v2.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5))

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        x = read_image(str(self.files[i]), mode=ImageReadMode.RGB).float() / 255.0  # (3, H, W)
        x = F.interpolate(x[None], size=self.out_hw, mode="bilinear",
                          align_corners=False, antialias=True)[0]

        if self.train:
            x = self.jitter(x)
            if torch.rand(1).item() < 0.3:
                x = self.blur(x)
            if torch.rand(1).item() < 0.3:
                x = x + torch.randn_like(x) * 0.01
            x = x.clamp(0.0, 1.0)

        return {"rgb": x, "idx": i, "name": self.files[i].stem}


def make_loader(dataset, batch_size=8, shuffle=False, num_workers=4):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True,
                      persistent_workers=num_workers > 0)



### Defining the DINO v2 Architecture

In [ ]:
import torch.nn as nn


class DinoV2Backbone(nn.Module):
    def __init__(self, variant="dinov2_vits14"):
        super().__init__()
        self.model = torch.hub.load("facebookresearch/dinov2", variant).eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    @torch.no_grad()
    def forward(self, rgb):  # (B, 3, H, W), float in [0, 1], H and W multiples of 14
        B, _, H, W = rgb.shape
        x = (rgb - self.mean) / self.std
        tokens = self.model.forward_features(x)["x_norm_patchtokens"]  # (B, N, C)
        return tokens.transpose(1, 2).reshape(B, -1, H // 14, W // 14)  # (B, C, H/14, W/14)